In [1]:
from collections.abc import Callable

import equinox as eqx
import jax
import jax.numpy as jnp
from context_flux_no.nn.embedding import PatchEmbedding
from context_flux_no.nn.misc import apply_componentwise, to_complex_dtype, to_ntuple
from einops import rearrange
from equinox._misc import default_floating_dtype
from jaxtyping import Array, Complex, Float, Int, PRNGKeyArray


In [74]:
def valid_frequency_inds(
    fft_shape: tuple[int, ...], frequency_modes: tuple[int, ...]
) -> tuple[Int[Array, "..."], ...]:
    slices = []
    for n, f_max in zip(fft_shape[:-1], frequency_modes[:-1]):
        if f_max >= n // 2:
            slices.append(jnp.r_[0:n])
        else:
            slices.append(jnp.r_[0 : f_max + 1, n - f_max : n])
    if frequency_modes[-1] >= fft_shape[-1]:
        slices.append(jnp.r_[0 : fft_shape[-1]])
    else:
        slices.append(jnp.r_[0 : frequency_modes[-1] + 1])
    return jnp.ix_(*slices)


class AFNO1D(eqx.Module):
    def __call__(self, x):
        return x


class AFNO2D(eqx.Module):
    per_block_mlp: eqx.nn.MLP

    in_channels: int = eqx.field(static=True)
    out_channels: int = eqx.field(static=True)
    max_frequency_modes: tuple[int, int] = eqx.field(static=True)
    num_blocks: int = eqx.field(static=True)

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        max_frequency_modes: int | tuple[int, int],
        num_blocks: int,
        hidden_channels: int,
        activation: Callable = jax.nn.gelu,
        dtype=None,
        *,
        key: PRNGKeyArray,
    ):
        if any(
            c % num_blocks != 0 for c in (in_channels, out_channels, hidden_channels)
        ):
            raise ValueError(
                """in_channels, out_channels, hidden_channels nust all be divisible by 
                num_blocks"""
            )

        dtype = default_floating_dtype() if dtype is None else dtype

        @eqx.filter_vmap
        def make_per_block_MLPs(key):
            return eqx.nn.MLP(
                in_size=in_channels // num_blocks,
                out_size=out_channels // num_blocks,
                width_size=hidden_channels // num_blocks,
                depth=1,
                activation=apply_componentwise(activation),
                dtype=to_complex_dtype(dtype),
                key=key,
            )

        self.per_block_mlp = make_per_block_MLPs(jax.random.split(key, num_blocks))

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.max_frequency_modes = to_ntuple(max_frequency_modes, 2)
        self.num_blocks = num_blocks

    @property
    def in_blocksize(self) -> int:
        return self.in_channels // self.num_blocks

    @property
    def out_blocksize(self) -> int:
        return self.out_channels // self.num_blocks

    def __call__(
        self, x: Float[Array, "in_channels grid_x grid_y"]
    ) -> Float[Array, "out_channels grid_x grid_y"]:
        x_fft = jnp.fft.rfftn(x, axes=tuple(range(1, x.ndim)))

        with jax.ensure_compile_time_eval():
            freq_mask = valid_frequency_inds(x_fft.shape, self.max_frequency_modes)
        x_fft_masked = x_fft[:, *freq_mask]
        x_fft_masked = rearrange(
            x_fft_masked, "(n b_in) fx fy -> n b_in fx fy", n=self.num_blocks
        )

        @eqx.filter_vmap
        def apply_mlp_per_block(mlp, z: Complex[Array, " in_blocksize *freqs"]):
            in_shape = z.shape
            out_shape = (self.out_blocksize,) + in_shape[1:]

            z = jnp.reshape(z, shape=(in_shape[0], -1))
            z_out = eqx.filter_vmap(mlp, in_axes=-1)(z)
            return jnp.reshape(z_out, shape=out_shape)

        y_fft_masked = apply_mlp_per_block(self.per_block_mlp, x_fft_masked)
        y_fft_masked = rearrange(y_fft_masked, "n b_out fx fy -> (n b_out) fx fy")

        y_fft = jnp.zeros_like(x_fft).at[:, *freq_mask].set(y_fft_masked)
        y = jnp.fft.irfftn(y_fft, s=x.shape[1:])

        # If out_channels is not in_channels, residual connection will not make sense
        return y + x

In [ ]:
def get_grid_1d(x: Float[Array, "... dim"]) -> Float[Array, "... 1"]:
    *rest, n_x, _ = x.shape
    grid_x = jnp.expand_dims(jnp.linspace(0, 1, n_x), axis=-1)
    return jnp.broadcast_to(grid_x, shape=(*rest, n_x, 1))


# TODO: move this to nn.positional_encoding.py and include sinusoidal embedding as part
# of the type hierarchy
class AbstractPositionEncoding(eqx.Module):
    pass


class LearnedPositionEncoding(AbstractPositionEncoding):
    encodings: Float[Array, "*shapes embedding_dim"]

    def __init__(self, position_shape: tuple[int, ...], encoding_dim: int):
        self.encodings = jnp.zeros(
            (*position_shape, encoding_dim),
        )

    def __call__(self, x: Float[Array, "*shapes embedding_dim"]):
        # TODO: need to assert the input shape matches the position_embedding
        return x + self.encodings


class DPOTBlock1D(eqx.Module):
    norm1: eqx.nn.GroupNorm
    norm2: eqx.nn.GroupNorm
    spatial_mixing: AFNO1D
    channel_mixing: eqx.nn.Sequential

    double_skip: bool = eqx.field(static=True)

    def __init__(
        self,
        channels: int,
        channels_hidden: int,
        groups: int = 8,
        double_skip: bool = True,
        activation: Callable = jax.nn.gelu,
        dtype=None,
        *,
        key: PRNGKeyArray,
    ):
        self.norm1 = eqx.nn.GroupNorm(groups, channels, dtype=dtype)
        self.norm2 = eqx.nn.GroupNorm(groups, channels, dtype=dtype)

        keys = jax.random.split(key, 3)
        self.channel_mixing = eqx.nn.Sequential(
            [
                eqx.nn.Conv(
                    num_spatial_dims=1,
                    in_channels=channels,
                    out_channels=channels_hidden,
                    kernel_size=1,
                    stride=1,
                    dtype=dtype,
                    key=keys[1],
                ),
                eqx.nn.Lambda(jax.nn.gelu),
                eqx.nn.Conv(
                    num_spatial_dims=1,
                    in_channels=channels_hidden,
                    out_channels=channels,
                    kernel_size=1,
                    stride=1,
                    dtype=dtype,
                    key=keys[2],
                ),
            ]
        )
        self.double_skip = double_skip

    def __call__(
        self,
        x: Float[Array, "channels patch_x"],
    ) -> Float[Array, "channels patch_x"]:
        y = self.spatial_mixing(self.norm1(x))

        if self.double_skip:
            y = y + x
            x = y

        y = self.channel_mixing(self.norm2(y))
        y = y + x
        return y


class DPOT1D(eqx.Module):
    patch_embedding: PatchEmbedding
    position_embedding: AbstractPositionEncoding
    time_aggregator: TimeAggregator
    blocks: list[DPOTBlock1D]
    output_head: eqx.nn.Sequential
    cls_head: eqx.nn.Sequential

    def __call__(
        self, u: Float[Array, "time channels grid_x"]
    ) -> Float[Array, "channels grid_x"]:
        # TODO: Check if normalization is actually used and implement if necessary
        grid = get_grid_1d(u)
        u: Float[Array, "time channels+1 grid_x"] = jnp.concatenate((u, grid), axis=-1)

        # Patch embedding for the spatial dimensions
        v: Float[Array, "time channels_embed patch_x"] = eqx.filter_vmap(
            self.patch_embedding
        )(u)

        # Apply positional embedding
        v = eqx.filter_vmap(self.position_embedding)(v)

        # Time aggregation layer
        v: Float[Array, "channels_embed patch_x"] = self.time_aggregator(v)

        for dpot_block in self.blocks:
            v = dpot_block(v)

        u_next = self.output_head(v)

        cls_token = jnp.mean(v, axis=-1)
        cls_pred = self.cls_head(cls_token)
        return u_next, cls_pred

NameError: name 'TimeAggregator' is not defined

In [75]:
afno2d = AFNO2D(
    in_channels=32,
    out_channels=32,
    max_frequency_modes=(7, 7),
    num_blocks=8,
    hidden_channels=64,
    key=jax.random.key(0),
)

In [76]:
test_img = jnp.ones((32, 64, 64))
eqx.filter_jit(afno2d)(test_img)[0, 0]

Array([ 0.58442324,  0.5053876 ,  0.48975646,  0.54331625,  0.66470873,
        0.8450774 ,  1.0684339 ,  1.3128486 ,  1.5524336 ,  1.7599211 ,
        1.9095068 ,  1.9795542 ,  1.9548137 ,  1.8279264 ,  1.6001372 ,
        1.2812862 ,  0.88920116,  0.44861192, -0.01036084, -0.4541316 ,
       -0.8480003 , -1.1588597 , -1.3580766 , -1.4242401 , -1.3453975 ,
       -1.1204238 , -0.7593045 , -0.28229904,  0.2818423 ,  0.89830166,
        1.5289124 ,  2.1348844 ,  2.6793654 ,  3.1297705 ,  3.459875  ,
        3.651626  ,  3.6965694 ,  3.5966911 ,  3.3644466 ,  3.021829  ,
        2.5984426 ,  2.1287794 ,  1.6490278 ,  1.1938616 ,  0.79360825,
        0.47209305,  0.24527246,  0.12062383,  0.09716761,  0.16599745,
        0.31124532,  0.5114778 ,  0.74153763,  0.9748038 ,  1.1857337 ,
        1.3524244 ,  1.4588516 ,  1.4964435 ,  1.4647417 ,  1.3710824 ,
        1.2294079 ,  1.0584543 ,  0.87962794,  0.7148317 ], dtype=float32)

In [27]:
freqs = jnp.meshgrid(jnp.fft.rfftfreq(6), jnp.fft.fftfreq(5))
freqs

[Array([[0.        , 0.16666667, 0.33333334, 0.5       ],
        [0.        , 0.16666667, 0.33333334, 0.5       ],
        [0.        , 0.16666667, 0.33333334, 0.5       ],
        [0.        , 0.16666667, 0.33333334, 0.5       ],
        [0.        , 0.16666667, 0.33333334, 0.5       ]], dtype=float32),
 Array([[ 0. ,  0. ,  0. ,  0. ],
        [ 0.2,  0.2,  0.2,  0.2],
        [ 0.4,  0.4,  0.4,  0.4],
        [-0.4, -0.4, -0.4, -0.4],
        [-0.2, -0.2, -0.2, -0.2]], dtype=float32)]

In [28]:
freqs[0].shape

(5, 4)

In [46]:
idx = valid_frequency_inds((5, 4), (1, 2))
idx

(Array([[0],
        [1],
        [4]], dtype=int32),
 Array([[0, 1, 2]], dtype=int32))

In [47]:
freqs[1][*idx]

Array([[ 0. ,  0. ,  0. ],
       [ 0.2,  0.2,  0.2],
       [-0.2, -0.2, -0.2]], dtype=float32)

In [48]:
freqs[0][*idx]

Array([[0.        , 0.16666667, 0.33333334],
       [0.        , 0.16666667, 0.33333334],
       [0.        , 0.16666667, 0.33333334]], dtype=float32)

In [22]:
jnp.ix_(jnp.r_[0:2, 4:4], jnp.r_[0:2])

(Array([[0],
        [1]], dtype=int32),
 Array([[0, 1]], dtype=int32))

In [16]:
x = jnp.ones((1, 2, 3))
tuple(range(1, x.ndim))

(1, 2)

In [6]:
x_complex = jax.random.normal(jax.random.key(0), (100,), dtype=jnp.complex_)
y = jnp.sin(x_complex)
jnp.array_equal(jnp.real(y), jnp.sin(jnp.real(x_complex)))

/tmp/ipykernel_2706436/1222268097.py:1: UserWarning: Explicitly requested dtype <class 'jax.numpy.complex128'>  is not available, and will be truncated to dtype complex64. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  x_complex = jax.random.normal(jax.random.key(0), (100,), dtype=jnp.complex_)


Array(False, dtype=bool)

In [12]:
y2 = separately_apply_real_imag(jax.nn.gelu)(x_complex)

In [13]:
jnp.array_equal(jnp.real(y2), jax.nn.gelu(jnp.real(x_complex)))

Array(True, dtype=bool)

In [18]:
a = (1, 2, 3)
(10, *a[1:])

(10, 2, 3)